# Prever o retorno de investimentos com Temporal Fusion Transformers

In [2]:
# Instala o pacote watermark. 
!pip install -q -U watermark

In [3]:
%env TF_CPP_MIN_LOG_LEVEL=3

env: TF_CPP_MIN_LOG_LEVEL=3


In [4]:
# https://www.tensorflow.org/
!pip install -q tensorflow

In [5]:
# https://pypi.org/project/ta/
!pip install -q ta

In [6]:
# https://pypi.org/project/yfinance/
!pip install -q yfinance

In [7]:
# Imports
import ta
import sklearn
import pandas as pd
import numpy as np
import tensorflow
import yfinance as yf
import matplotlib.pyplot as plt
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")

## Extração dos dados

In [14]:
def download_data(ticker, start = "2000-01-01", end = "2024-12-31"):
    
    # Força que venha 'Adj Close' definindo auto_adjust=False
    dados = yf.download(ticker, start=start, end=end, auto_adjust=False)

    # Mapeamento dos nomes originais para minúsculas e underscore
    mapeamento = {
        'Open':       'open',
        'High':       'high',
        'Low':        'low',
        'Close':      'close',
        'Adj Close':  'adj_close',
        'Volume':     'volume'
    }

    presentes = {col: novo for col, novo in mapeamento.items() if col in dados.columns}
    dados.rename(columns = presentes, inplace = True)

    dados.index.name = "date"
    
    return dados

In [10]:
# Extração dos dados
df = download_data("MSFT")

[*********************100%***********************]  1 of 1 completed


In [11]:
df2 = download_data("petr4.SA")

[*********************100%***********************]  1 of 1 completed


In [20]:
df.head()

Price,adj_close,close,high,low,open,volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,MSFT
date,,,,,,
2000-01-03,35.601452,58.28125,59.3125,56.00000,58.68750,53228400
2000-01-04,34.398811,56.31250,58.5625,56.12500,56.78125,54119000
2000-01-05,34.761528,56.90625,58.1875,54.68750,55.56250,64059600
2000-01-06,33.597073,55.00000,56.9375,54.18750,56.09375,54976600
2000-01-07,34.036140,55.71875,56.1250,53.65625,54.31250,62013600


In [21]:
df.tail()

Price,adj_close,close,high,low,open,volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,MSFT
date,,,,,,
2023-12-22,369.077087,374.579987,375.179993,372.709991,373.679993,17107500
2023-12-26,369.155914,374.660004,376.940002,373.500000,375.000000,12673100
2023-12-27,368.574615,374.070007,375.059998,372.809998,373.690002,14905400
2023-12-28,369.766846,375.279999,376.459991,374.160004,375.369995,14327000
2023-12-29,370.515686,376.040009,377.160004,373.480011,376.000000,18730800


## Engenharia de Atributos

Pacote `ta`: Technical Analysis Library in Python

https://github.com/bukosabino/ta

Criaremos uma função de engenharia de atributos para um dataframe que representa dados de um ativo financeiro (ações) com colunas para 'open' (preço de abertura), 'high' (máximo do dia), 'low' (mínimo do dia), 'close' (preço de fechamento) e 'volume'. 

A engenharia de atributos é uma técnica usada para criar novas variáveis com base em variáveis existentes, a fim de melhorar o desempenho dos modelos de Machine Learning. 

In [ ]:
# Função para engenharia de atributos com dados coletados na versão mais recente do yfinance
def func_engenharia_atributos(df):

    df_copy = df.copy()
    
    # Se houver MultiIndex nas colunas, mantém só o primeiro nível
    if isinstance(df_copy.columns, pd.MultiIndex):
        df_copy.columns = df_copy.columns.get_level_values(0)
    
    # Cria a variável com o retorno (mudança percentual do fechamento - close)
    # Essa será nossa variável alvo
    df_copy["retorno"] = df_copy["close"].pct_change(1)
    
    # Shift das colunas de preço do ativo financeiro
    df_copy["op"]  = df_copy["open"].shift(1)
    df_copy["hi"]  = df_copy["high"].shift(1)
    df_copy["lo"]  = df_copy["low"].shift(1)
    df_copy["clo"] = df_copy["close"].shift(1)

    # Shift da coluna Volume
    df_copy["vol"] = df_copy["volume"].shift(1)
    
    # Simple Moving Average (SMA)
    df_copy["SMA 15"] = df_copy["close"].rolling(15).mean().shift(1)
    df_copy["SMA 60"] = df_copy["close"].rolling(60).mean().shift(1)

    # Moving Standard Deviation (MSD) - Volatilidade
    df_copy["MSD 15"] = df_copy["retorno"].rolling(15).std().shift(1)
    df_copy["MSD 60"] = df_copy["retorno"].rolling(60).std().shift(1)
    
    # Volume Weighted Average Price (VWAP)
    vwap = ta.volume.VolumeWeightedAveragePrice(high   = df_copy["high"],
                                                low    = df_copy["low"],
                                                close  = df_copy["close"],
                                                volume = df_copy["volume"],
                                                window = 5)
    
    df_copy["VWAP"] = vwap.vwap.shift(1)
    
    # RSI (agora df_copy["close"] é Series 1D)
    rsi = ta.momentum.RSIIndicator(df_copy["close"], window = 5, fillna = False)
    df_copy["RSI"] = rsi.rsi().shift(1)
    
    return df_copy.dropna()

`func_engenharia_atributos`

- Cria uma cópia do dataframe para não modificar o dataframe original.

- Cria uma nova coluna 'retorno', que representa a mudança percentual do preço de fechamento em relação ao dia anterior.

- As colunas 'open', 'high', 'low', 'close' e 'volume' são então deslocadas para baixo (shifted) em uma unidade, criando as colunas 'op', 'hi', 'lo', 'clo' e 'vol'. Isso significa que a linha 'i' dessas novas colunas contém os valores da linha 'i-1' das colunas originais. Isso é chamado de defasagem e amplamente usado na modelagem de séries temporais.

- Cria médias móveis simples (Simple Moving Average - SMA) para o preço de fechamento com janelas de 15 e 60 dias, deslocadas por uma unidade. 

- A volatilidade, representada como o desvio padrão das mudanças percentuais do preço de fechamento, é calculada para as janelas de 15 e 60 dias, também deslocadas por uma unidade. Esse índice é conhecido como Moving Standard Deviation (MSD).

- Cria o Volume Weighted Average Price (VWAP) com uma janela de 5 dias. O VWAP é uma medida de preço médio ponderado pelo volume.

- Adiciona o indicador RSI (Relative Strength Index) com uma janela de 5 dias. O RSI é um indicador de momento que mede a velocidade e a mudança de movimentos de preço.

- Remove todas as linhas que contêm valores NA (que foram criados ao deslocar as colunas e calcular médias móveis e RSI com janelas).

Em resumo, essa função cria várias novas características técnicas comumente usadas na análise de ativos financeiros, todas deslocadas por uma unidade, para evitar o uso de informações futuras (ou seja, vazamento de dados) no modelo de Machine Learning.